## <b><font color='darkblue'>Preface</font></b>
This notebook is used to demonstrate the API usage of package **[`Cockatoo.AI`](https://github.com/Cockatoo-AI-Org/Cockatoo.AI)**.

### <b><font color='darkgreen'>Import Packages</font></b>
Firstly, let's import the necessary pacakges:

In [28]:
from IPython.display import Audio
import os
from cockatoo_ai.utils import wrapper
from cockatoo_ai.utils import model_a
import warnings
warnings.filterwarnings('ignore')

Audio2TextData = wrapper.Audio2TextData  # ModelA's output
LangEnum = wrapper.LangEnum  # Supported language types. e.g. en, cn, multi

### <b><font color='darkgreen'>Testing dataset</font></b>
We hold the testing dataset under path `~/Projects/model_a/test_data/` ([source](https://drive.google.com/drive/folders/17JKgdMwIW76aD6iePij_jCiU1dRrAaYH)). Below we will prepare two variables link to audio files for testing.

In [2]:
MODEL_A_TEST_DATA_ROOT = os.path.expanduser('~/Projects/model_a/test_data/')
MODEL_A_TEST_EN_AUDIO_FILE_PATH = os.path.join(MODEL_A_TEST_DATA_ROOT, 'en_20240108_johnlee.wav')
MODEL_A_TEST_CN_AUDIO_FILE_PATH = os.path.join(MODEL_A_TEST_DATA_ROOT, 'cn_20240108_johnlee.wav')

In [3]:
!ls -hl ~/Projects/model_a/test_data/

total 5.9M
-rw-r--r-- 1 root root   79 Mar  9 02:00 cn_20240108_johnlee.txt
-rw-r--r-- 1 root root 753K Mar  9 01:56 cn_20240108_johnlee.wav
-rw-rw-r-- 1 john john  110 Mar  9 02:00 en_20240108_johnlee.txt
-rw-r--r-- 1 root root 2.5M Mar  9 01:56 en_20240108_johnlee.wav
-rw-r--r-- 1 root root 2.7M Mar  9 01:56 en_20240316_demmi.wav


In [5]:
# Get the output of audio file (en) as ground truth:
output_lines = !cat ~/Projects/model_a/test_data/en_20240108_johnlee.txt
TEST_EN_AUDIO_SCRIPT = output_lines[0]
TEST_EN_AUDIO_SCRIPT

'Hello, this is for testing in English. We will use this to evaluate model SST and see how it performs. Thanks.'

In [6]:
# Get the output of audio file (cn) as ground truth:
output_lines = !cat ~/Projects/model_a/test_data/cn_20240108_johnlee.txt
TEST_CN_AUDIO_SCRIPT = output_lines[0]
TEST_CN_AUDIO_SCRIPT

'中文測試. 這的檔案是使用來檢視模型 SST 的轉換效果. 謝謝.'

In [6]:
Audio(filename=MODEL_A_TEST_EN_AUDIO_FILE_PATH)

In [7]:
Audio(filename=MODEL_A_TEST_CN_AUDIO_FILE_PATH)

### <b><font color='darkgreen'>Environment Variables</font></b>
Some models require certain environment variables to be set beforehand (e.g. [`OpenAI API Key`](https://platform.openai.com/docs/api-reference/authentication).).  Please create a `~/.env` file to store these environment variables, ensuring that future models function properly.

In [7]:
# List the given environment variables:
!cut -d= -f1 ~/.env

OPENAI_API_KEY
GCP_PROJECT_ID
GCP_LOCATION
GCP_BUCKET
GOOGLE_API_KEY
GOOGLE_APPLICATION_CREDENTIALS


## <b><font color='darkblue'>Model A</font> (Speech to text, STT)</b>
For **Model A** (Speech to text, STT), we defined an abstraction as `wrapper.ModelA`:

In [9]:
help(wrapper.ModelA)

Help on class ModelA in module cockatoo_ai.utils.wrapper:

class ModelA(typing.Protocol)
 |  ModelA(lang: cockatoo_ai.utils.wrapper.LangEnum = <LangEnum.en: 0>)
 |
 |  Model A wrapper.
 |
 |  Method resolution order:
 |      ModelA
 |      typing.Protocol
 |      typing.Generic
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, lang: cockatoo_ai.utils.wrapper.LangEnum = <LangEnum.en: 0>)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  audio_2_text(self, audio_file_path: str) -> cockatoo_ai.utils.wrapper.Audio2TextData
 |      Turns audio of given file path into text.
 |
 |      Args:
 |        audio_file_path: Audio file path to do audio to text transformation.
 |
 |      Returns:
 |        `Audio2TextData` with trasnformed text.
 |
 |  live_2_text(self, record_time_sec: int = 5, output_audio_file_path: str | None = None) -> cockatoo_ai.utils.wrapper.Audio2TextData
 |      Records and transform audio into text.
 |
 |      Args:
 |  

We will then introduce how to leverage **[`Cockatoo.AI`](https://github.com/Cockatoo-AI-Org/Cockatoo.AI)** APIs to achive the tasks of Model A below.

### <b><font color='darkgreen'>Package `cockatoo_ai.utils.model_a`</font></b>
This package provides instances of Model A implementations. The supported Model A implementations are listed in the **`model_a.ModelType enum`**.

In [8]:
ModelAType = model_a.ModelType

In [9]:
print('\n'.join([f'{t.name}/{t.value}' for t in ModelAType]))

OPEN_AI_WHISPER_OFFLINE/open_ai_whisper_offline
SR_OPEN_AI_WHISPER/sr_open_ai_whisper
SR_GCP/sr_gcp
GCP_SPEECH_2_TEXT/gcp_speech_2_text


We will demonstrate the usage of them from next few sub sections. This package provides a method `get` to obtain Model A solution:

In [12]:
help(model_a.get)

Help on function get in module cockatoo_ai.utils.model_a:

get(model_type: cockatoo_ai.utils.model_a.ModelType | str, settings: dict[str, typing.Any] | None = None) -> cockatoo_ai.utils.wrapper.ModelA
    Gets model A.

    Args:
      model_type: Model type.
      settings: Model settings.

    Returns:
      Model A wrapper implementation.



Then let's see how it works.

#### <b><font size='3ptx'>`model_a.ModelType.SR_OPEN_AI_WHISPER`</font></b>
This option is coming from [**speech_recognition**.Recognizer](https://pypi.org/project/SpeechRecognition/#description) (for [OpenAI Whisper API](https://platform.openai.com/docs/guides/speech-to-text) users).
- **Type:** Remote API
- **Supported languages:** `en`, `ch`
- **Usage limitation:** Need OpenAI's API key and set it in environment variable `OPENAI_API_KEY=...`

In [10]:
ma_openai_whisper_wp = model_a.get('sr_open_ai_whisper')
ma_openai_whisper_wp

Then we could call method `audio_2_text` to transform the input audio file into text:

In [11]:
%%time
model_a_result = ma_openai_whisper_wp.audio_2_text(MODEL_A_TEST_EN_AUDIO_FILE_PATH)

CPU times: user 86.2 ms, sys: 120 ms, total: 206 ms
Wall time: 4.22 s


The returned result will contain:
- **text**: The transformed text from audio file or waves.
- **spent_time_sec**: The time required to complete the task.
- **audio_file_path**: The input audio file path.

In [12]:
model_a_result

Audio2TextData(text='Hello, this is for testing in English. We will use this to evaluate model SST and see how it performs. Thanks.', spent_time_sec=4.220035076141357, audio_file_path='/root/Projects/model_a/test_data/en_20240108_johnlee.wav')

In [13]:
print(model_a_result.text)

Hello, this is for testing in English. We will use this to evaluate model SST and see how it performs. Thanks.


In [14]:
print(TEST_EN_AUDIO_SCRIPT)

Hello, this is for testing in English. We will use this to evaluate model SST and see how it performs. Thanks.


In [20]:
output = ma_openai_whisper_wp.live_2_text(record_time_sec=7)

ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.rear
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.center_lfe
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.side
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.surround71
ALSA lib setup.c:547:(add_elem) Cannot obtain info for CTL elem (MIXER,'IEC958 Playback Default',0,0,0): No such file or directory
ALSA lib setup.c:547:(add_elem) Cannot obtain info for CTL elem (MIXER,'IEC958 Playback Default',0,0,0): No such file or directory
ALSA lib setup.c:547:(add_elem) Cannot obtain info for CTL elem (MIXER,'IEC958 Playback Default',0,0,0): No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.hdmi
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.hdmi
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.modem
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM cards.pcm.modem
ALSA lib pcm.c:2721:(snd_

In [21]:
output

Audio2TextData(text='这双文字', spent_time_sec=2.7551076412200928, audio_file_path='/tmp/modela_live_audio.wav')

In [17]:
%%time
model_a_result = ma_openai_whisper_wp.audio_2_text(MODEL_A_TEST_CN_AUDIO_FILE_PATH)

CPU times: user 11.8 ms, sys: 8.21 ms, total: 20 ms
Wall time: 2.1 s


In [18]:
print(model_a_result.text)

中文測試,這個檔案是使用來檢視模型SSD的轉換效果,謝謝


In [19]:
print(TEST_CN_AUDIO_SCRIPT)

中文測試. 這的檔案是使用來檢視模型 SST 的轉換效果. 謝謝.


#### <b><font size='3ptx'>`model_a.ModelType.SR_GCP`</font></b>
This option is coming from [**speech_recognition**.Recognizer](https://pypi.org/project/SpeechRecognition/#description) (for [GCP Speech to text API](https://cloud.google.com/speech-to-text?hl=zh_tw) users). According to the testing result, it is broken right now.
- **Type:** Remote API
- **Supported language:** `en`, `cn` and [more](https://cloud.google.com/speech-to-text?hl=zh_tw)
- **Usage limitation:** Need GCP API key and set it in environment variable `GOOGLE_API_KEY=...`

In [22]:
ma_gcp_wp = model_a.get('sr_gcp')
ma_gcp_wp

In [24]:
#%%time
#model_a_result = ma_gcp_wp.audio_2_text(MODEL_A_TEST_EN_AUDIO_FILE_PATH)

In [25]:
#print(model_a_result.text)

In [26]:
#%%time
#ma_gcp_wp = model_a.get('sr_gcp', {'lang': LangEnum.CN})
#model_a_result = ma_gcp_wp.audio_2_text(MODEL_A_TEST_CN_AUDIO_FILE_PATH)

In [27]:
#print(model_a_result.text)

#### <b><font size='3ptx'>`model_a.ModelType.GCP_SPEECH_2_TEXT`</font></b>
This option is the wrapper of [**GCP Speech-to-text**](https://cloud.google.com/speech-to-text?hl=zh_tw) solutions.
- **Type:** Remote API
- **Supported languages:** `en`, `cn` and [more](https://cloud.google.com/speech-to-text?hl=zh_tw)
- **Usage limitation:** Need GCP API key and set it in environment variable `GOOGLE_API_KEY=...`

In [28]:
ma_gcp_speech_2_text_wp = model_a.get(ModelAType.GCP_SPEECH_2_TEXT)

In [29]:
%%time
model_a_result = ma_gcp_speech_2_text_wp.audio_2_text(MODEL_A_TEST_EN_AUDIO_FILE_PATH)

CPU times: user 102 ms, sys: 61.7 ms, total: 163 ms
Wall time: 3.58 s


In [30]:
print(model_a_result.text)

Hello, this is for testing in English, we will use this. To evaluate model SST and see how it performs, thanks.


In [31]:
ma_gcp_speech_2_text_wp = model_a.get(ModelAType.GCP_SPEECH_2_TEXT, {'lang': LangEnum.cn})

In [32]:
%%time
#model_a_result = ma_gcp_speech_2_text_wp.audio_2_text(MODEL_A_TEST_CN_AUDIO_FILE_PATH)

CPU times: user 6 μs, sys: 6 μs, total: 12 μs
Wall time: 22.9 μs


#### <b><font size='3ptx'>`model_a.ModelType.OPEN_AI_WHISPER_OFFLINE`</font></b>
This option is the wrapper of [**GCP Speech-to-text**](https://github.com/openai/whisper) solutions.
- **Type:** Local
- **Supported languages:** `en`
- **Usage limitation:** This solution will download the model to local. So the disk and memory usage can be limitation. Also, the process time will vary according to the hardware spec as well.

![model size](images/whisper_size.png)

In [23]:
ma_whisper_offline_wp = model_a.get(ModelAType.OPEN_AI_WHISPER_OFFLINE)

In [24]:
%%time
model_a_result = ma_whisper_offline_wp.audio_2_text(MODEL_A_TEST_EN_AUDIO_FILE_PATH)

/home/john/Gitrepos/Cockatoo.AI/env/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


CPU times: user 1min 39s, sys: 6.11 s, total: 1min 45s
Wall time: 39.2 s


In [25]:
print(model_a_result.text)

 Hello, this is for testing in English. We will use this to evaluate model SST and see how it performs. Thanks.


In [26]:
ma_whisper_offline_wp = model_a.get(ModelAType.OPEN_AI_WHISPER_OFFLINE, {'lang': LangEnum.cn})

In [27]:
%%time
model_a_result = ma_whisper_offline_wp.audio_2_text(MODEL_A_TEST_CN_AUDIO_FILE_PATH)

/home/john/Gitrepos/Cockatoo.AI/env/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


CPU times: user 1min 10s, sys: 8.6 s, total: 1min 19s
Wall time: 22.6 s


In [38]:
print(model_a_result.text)

中文測試,這個檔案是使用來檢視模型ASST的轉換效果謝謝
